# Notebook 3 — Train / Validation / Test Split

## Objective

Split the labeled dataset into training, validation, and test sets before performing
deep analysis or making modeling decisions.

Because delivery performance can change over time, a time-based split is used.
This allows the model to learn from historical orders and evaluate its performance
on later orders, which better represents a real-world prediction scenario.

## Split

- Train: earliest 70% of orders
- Validation: next 15%
- Test: latest 15%

The label distribution is checked in each split to understand whether the severe
class imbalance is consistent across the datasets.

## Artifact

`artifacts/train.parquet`

`artifacts/validation.parquet`

`artifacts/test.parquet`

# Load the labeled data

In [1]:
import pandas as pd

labeled_table = pd.read_parquet(
    "../artifacts/labeled_table.parquet"
)

print("Shape:", labeled_table.shape)
display(labeled_table.head())

Shape: (96476, 20)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,item_count,product_count,seller_count,total_price,total_freight,payment_count,payment_value,max_installments,label
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,3.0,38.71,1.0,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,BA,1.0,1.0,1.0,118.70,22.76,1.0,141.46,1.0,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,1.0,179.12,3.0,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,1.0,72.20,1.0,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,1.0,1.0,1.0,19.90,8.72,1.0,28.62,1.0,On Time


# Prepare the date

In [2]:
labeled_table["order_purchase_timestamp"] = pd.to_datetime(
    labeled_table["order_purchase_timestamp"],
    errors="coerce"
)

In [3]:
print(
    "Date range:",
    labeled_table["order_purchase_timestamp"].min(),
    "to",
    labeled_table["order_purchase_timestamp"].max()
)

Date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37


In [4]:
print(
    "Missing purchase dates:",
    labeled_table["order_purchase_timestamp"].isna().sum()
)

Missing purchase dates: 0


In [5]:
labeled_table = labeled_table.dropna(
    subset=["order_purchase_timestamp"]
).copy()

# Sort chronologically

In [6]:
labeled_table = labeled_table.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

# Create the time-based split

In [7]:
n = len(labeled_table)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train = labeled_table.iloc[:train_end].copy()
validation = labeled_table.iloc[train_end:validation_end].copy()
test = labeled_table.iloc[validation_end:].copy()

# Check the date range of each split

In [8]:
def check_split(name, df):
    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(df))
    print(
        "Date range:",
        df["order_purchase_timestamp"].min(),
        "to",
        df["order_purchase_timestamp"].max()
    )
    print("Label distribution:")
    print(
        df["label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

check_split("Train", train)
check_split("Validation", validation)
check_split("Test", test)


Train
----------------------------------------
Rows: 67533
Date range: 2016-09-15 12:16:38 to 2018-04-15 20:07:56
Label distribution:
label
On Time    90.97
Late        9.03
Name: proportion, dtype: float64

Validation
----------------------------------------
Rows: 14471
Date range: 2018-04-15 20:10:23 to 2018-06-21 07:50:39
Label distribution:
label
On Time    94.66
Late        5.34
Name: proportion, dtype: float64

Test
----------------------------------------
Rows: 14472
Date range: 2018-06-21 08:29:29 to 2018-08-29 15:00:37
Label distribution:
label
On Time    93.39
Late        6.61
Name: proportion, dtype: float64


# Check label balance

In [9]:
split_distribution = pd.DataFrame({
    "Train": train["label"].value_counts(normalize=True),
    "Validation": validation["label"].value_counts(normalize=True),
    "Test": test["label"].value_counts(normalize=True)
}).T * 100

split_distribution = split_distribution.round(2)

display(split_distribution)

label,On Time,Late
Train,90.97,9.03
Validation,94.66,5.34
Test,93.39,6.61


# Verify chronological separation

In [10]:
assert train["order_purchase_timestamp"].max() < validation["order_purchase_timestamp"].min()
assert validation["order_purchase_timestamp"].max() < test["order_purchase_timestamp"].min()

print("Splits are strictly chronological")

Splits are strictly chronological


# Verify no order appears in multiple splits

In [11]:
train_ids = set(train["order_id"])
validation_ids = set(validation["order_id"])
test_ids = set(test["order_id"])

print("Train/Validation overlap:", len(train_ids & validation_ids))
print("Train/Test overlap:", len(train_ids & test_ids))
print("Validation/Test overlap:", len(validation_ids & test_ids))

Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


In [12]:
assert len(train_ids & validation_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0

print("No order appears in more than one split")

No order appears in more than one split


# Save the artifacts

In [13]:
train.to_parquet(
    "../artifacts/train.parquet",
    index=False
)

validation.to_parquet(
    "../artifacts/validation.parquet",
    index=False
)

test.to_parquet(
    "../artifacts/test.parquet",
    index=False
)

print("Train, validation, and test artifacts saved")

Train, validation, and test artifacts saved


# Verify the saved files

In [14]:
train_saved = pd.read_parquet("../artifacts/train.parquet")
validation_saved = pd.read_parquet("../artifacts/validation.parquet")
test_saved = pd.read_parquet("../artifacts/test.parquet")

print("Train:", train_saved.shape)
print("Validation:", validation_saved.shape)
print("Test:", test_saved.shape)

Train: (67533, 20)
Validation: (14471, 20)
Test: (14472, 20)


In [15]:
assert train_saved["order_id"].is_unique
assert validation_saved["order_id"].is_unique
assert test_saved["order_id"].is_unique

print("Saved artifacts verified")

Saved artifacts verified


## Split Decision

A time-based split was selected instead of a random split because the goal is to
predict delivery performance for future orders. The model will therefore be
trained on earlier orders and evaluated on later orders, which better represents
a real-world prediction scenario.

The data was divided chronologically into 70% training, 15% validation, and 15%
test data.

The label distribution was checked across the three splits. The Late class is
the minority class in all splits, with 9.03% Late orders in the training set,
5.34% in validation, and 6.61% in the test set.

This confirms a significant class imbalance. The class distribution also changes
over time, so the model should not be evaluated using accuracy alone. Precision,
recall, F1-score, and other appropriate classification metrics should be
considered in later stages, with particular attention to identifying Late orders.

The splits are strictly chronological and contain no overlapping order IDs.